# vLLM: Fast and Easy LLM Serving

vLLM is a high-performance inference and serving engine for large language models (LLMs). It's designed to maximize throughput and minimize latency for LLM deployments.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Performance Optimization](#performance)
8. [Production Deployment](#deployment)
9. [Best Practices](#best-practices)
10. [Resources](#resources)

## Introduction

vLLM is an open-source library for fast LLM inference and serving. It implements several optimization techniques including:

- **PagedAttention**: Efficient memory management inspired by virtual memory paging
- **Continuous batching**: Dynamic batching of requests for higher throughput
- **Optimized CUDA kernels**: Fast attention mechanisms and sampling
- **Tensor parallelism**: Multi-GPU support for large models
- **Streaming outputs**: Real-time token generation

### Use Cases

- High-throughput LLM serving for production applications
- Real-time chatbots and conversational AI
- Batch inference workloads
- Multi-user LLM applications

## Key Features

| Feature | Description |
|---------|-------------|
| **PagedAttention** | Reduces memory waste and enables larger batch sizes |
| **Continuous Batching** | Processes requests dynamically without waiting for batch completion |
| **Fast Model Execution** | Optimized CUDA kernels for attention and sampling |
| **OpenAI-Compatible API** | Drop-in replacement for OpenAI API |
| **Streaming** | Token-by-token streaming for real-time responses |
| **Multi-GPU Support** | Tensor parallelism for large models |
| **Quantization** | Support for AWQ, GPTQ, and SqueezeLLM |

## Architecture Overview

```
┌─────────────────────────────────────────────┐
│           Client Applications               │
└─────────────────┬───────────────────────────┘
                  │
                  ▼
┌─────────────────────────────────────────────┐
│         vLLM API Server (FastAPI)           │
│  • OpenAI-compatible endpoints              │
│  • Request queuing and management           │
└─────────────────┬───────────────────────────┘
                  │
                  ▼
┌─────────────────────────────────────────────┐
│         LLM Engine                          │
│  • Continuous batching scheduler            │
│  • PagedAttention memory manager            │
└─────────────────┬───────────────────────────┘
                  │
                  ▼
┌─────────────────────────────────────────────┐
│         Model Execution                     │
│  • Optimized CUDA kernels                   │
│  • Tensor parallelism (multi-GPU)           │
│  • Quantization support                     │
└─────────────────────────────────────────────┘
```

## Installation

**Note**: For Google Colab, uncomment and run the installation cell below.

In [ ]:
# Uncomment to install vLLM in Colab
# !pip install vllm

## Basic Usage

### Offline Inference

The simplest way to use vLLM is for offline batch inference:

In [ ]:
from vllm import LLM, SamplingParams

# Initialize the model
llm = LLM(model="facebook/opt-125m")  # Using a small model for demo

# Define sampling parameters
sampling_params = SamplingParams(
    temperature=0.8,
    top_p=0.95,
    max_tokens=100
)

# Generate text
prompts = [
    "The future of AI is",
    "Machine learning helps us",
    "In the world of technology,"
]

outputs = llm.generate(prompts, sampling_params)

# Print results
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}")
    print(f"Generated: {generated_text!r}")
    print("-" * 80)

### OpenAI-Compatible Server

vLLM provides an OpenAI-compatible API server. To start the server:

```bash
python -m vllm.entrypoints.openai.api_server \
    --model facebook/opt-125m \
    --port 8000
```

Then use it with the OpenAI client:

In [ ]:
from openai import OpenAI

# Initialize client pointing to vLLM server
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="token-abc123"  # vLLM server doesn't require a real key
)

# Create a completion
completion = client.completions.create(
    model="facebook/opt-125m",
    prompt="San Francisco is a",
    max_tokens=50,
    temperature=0.7
)

print(completion.choices[0].text)

### Chat Completions

For chat models like Llama or Mistral:

In [ ]:
# Using the OpenAI client with chat completions
chat_completion = client.chat.completions.create(
    model="meta-llama/Llama-2-7b-chat-hf",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is machine learning?"}
    ],
    temperature=0.7,
    max_tokens=200
)

print(chat_completion.choices[0].message.content)

## Advanced Features

### Streaming Outputs

Generate tokens one at a time for real-time applications:

In [ ]:
# Streaming with the offline API
from vllm import LLM, SamplingParams

llm = LLM(model="facebook/opt-125m")
sampling_params = SamplingParams(temperature=0.8, max_tokens=100)

prompts = ["Tell me about artificial intelligence"]

# Generate with streaming
for output in llm.generate(prompts, sampling_params, use_tqdm=False):
    print(output.outputs[0].text)

In [ ]:
# Streaming with the OpenAI API
stream = client.completions.create(
    model="facebook/opt-125m",
    prompt="The benefits of cloud computing are",
    max_tokens=100,
    stream=True
)

for chunk in stream:
    if chunk.choices[0].text:
        print(chunk.choices[0].text, end='', flush=True)

### Multi-GPU Inference

Use tensor parallelism for large models across multiple GPUs:

In [ ]:
# Initialize with tensor parallelism
llm = LLM(
    model="meta-llama/Llama-2-70b-hf",
    tensor_parallel_size=4,  # Use 4 GPUs
    dtype="float16"
)

# Use as normal
outputs = llm.generate(prompts, sampling_params)

### Quantization

Load quantized models for reduced memory usage:

In [ ]:
# AWQ quantized model
llm = LLM(
    model="TheBloke/Llama-2-7B-AWQ",
    quantization="awq"
)

# GPTQ quantized model
llm = LLM(
    model="TheBloke/Llama-2-7B-GPTQ",
    quantization="gptq"
)

## Performance Optimization

### Key Parameters

Optimize performance by tuning these parameters:

In [ ]:
llm = LLM(
    model="facebook/opt-125m",
    # GPU memory utilization (0.0 - 1.0)
    gpu_memory_utilization=0.9,
    
    # Maximum number of sequences in a batch
    max_num_seqs=256,
    
    # Maximum number of batched tokens
    max_num_batched_tokens=8192,
    
    # Data type for model weights
    dtype="float16",
    
    # Enable CUDA graph for better performance
    enforce_eager=False,
)

### Benchmarking

Measure throughput and latency:

In [ ]:
import time

# Benchmark throughput
prompts = ["Test prompt"] * 100
sampling_params = SamplingParams(temperature=0.8, max_tokens=50)

start_time = time.time()
outputs = llm.generate(prompts, sampling_params)
end_time = time.time()

total_tokens = sum(len(output.outputs[0].token_ids) for output in outputs)
throughput = total_tokens / (end_time - start_time)

print(f"Throughput: {throughput:.2f} tokens/second")
print(f"Latency: {(end_time - start_time) / len(prompts):.4f} seconds/request")

## Production Deployment

### Docker Deployment

Example Dockerfile:

```dockerfile
FROM vllm/vllm-openai:latest

# Set model name
ENV MODEL_NAME=meta-llama/Llama-2-7b-chat-hf

# Expose port
EXPOSE 8000

# Run server
CMD ["python", "-m", "vllm.entrypoints.openai.api_server", \
     "--model", "${MODEL_NAME}", \
     "--host", "0.0.0.0", \
     "--port", "8000"]
```

### Kubernetes Deployment

Example Kubernetes manifest:

```yaml
apiVersion: v1
kind: Service
metadata:
  name: vllm-service
spec:
  selector:
    app: vllm
  ports:
  - port: 8000
    targetPort: 8000
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: vllm-deployment
spec:
  replicas: 2
  selector:
    matchLabels:
      app: vllm
  template:
    metadata:
      labels:
        app: vllm
    spec:
      containers:
      - name: vllm
        image: vllm/vllm-openai:latest
        resources:
          limits:
            nvidia.com/gpu: 1
        env:
        - name: MODEL_NAME
          value: "meta-llama/Llama-2-7b-chat-hf"
        ports:
        - containerPort: 8000
```

### Monitoring

vLLM exposes Prometheus metrics for monitoring:

In [ ]:
# Start server with metrics enabled
# python -m vllm.entrypoints.openai.api_server \
#     --model facebook/opt-125m \
#     --port 8000 \
#     --enable-metrics

# Metrics available at http://localhost:8000/metrics
# Key metrics:
# - vllm:num_requests_running
# - vllm:num_requests_waiting
# - vllm:gpu_cache_usage_perc
# - vllm:time_to_first_token_seconds
# - vllm:time_per_output_token_seconds

## Best Practices

1. **Memory Management**
   - Set `gpu_memory_utilization` to 0.9 for production workloads
   - Use quantization for large models to reduce memory footprint
   - Monitor GPU memory usage with `nvidia-smi`

2. **Performance Tuning**
   - Use tensor parallelism for models > 13B parameters
   - Enable CUDA graphs for faster inference (`enforce_eager=False`)
   - Tune `max_num_batched_tokens` based on your workload

3. **Production Deployment**
   - Use health check endpoints for load balancers
   - Implement request queuing for burst traffic
   - Monitor metrics for performance degradation
   - Use horizontal pod autoscaling in Kubernetes

4. **Model Selection**
   - Choose model size based on latency requirements
   - Consider quantized models for cost optimization
   - Test different models for your specific use case

5. **Error Handling**
   - Implement retry logic for transient failures
   - Set appropriate timeouts for long-running requests
   - Handle out-of-memory errors gracefully

## Resources

- **Official Documentation**: https://docs.vllm.ai/
- **GitHub Repository**: https://github.com/vllm-project/vllm
- **Paper**: "Efficient Memory Management for Large Language Model Serving with PagedAttention"
- **Community**: 
  - Discord: https://discord.gg/vllm
  - GitHub Discussions: https://github.com/vllm-project/vllm/discussions

### Related Technologies

- **TGI (Text Generation Inference)**: Hugging Face's inference server
- **TensorRT-LLM**: NVIDIA's optimized LLM inference
- **DeepSpeed-MII**: Microsoft's model serving solution
- **Ray Serve**: General-purpose model serving framework